# Part B — Question 3: Ablation Study
## Task 3.2: Failure Mode

**Student:** Prashant Kumar | **Roll:** 230101  
**Paper:** *Finding Deceptive Opinion Spam by Any Stretch of the Imagination* — Ott et al., ACL 2011  

This notebook demonstrates one scenario in which the BIGRAMS+_SVM method fails noticeably and connects that failure directly to **Assumption 1** identified in Task 1.2.

---
## Random Seed and Hyperparameter Declarations

In [1]:
RANDOM_SEED  = 42
import numpy as np
np.random.seed(RANDOM_SEED)
import warnings; warnings.filterwarnings('ignore')
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import os

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.metrics import (make_scorer, f1_score, recall_score,
                              precision_score, confusion_matrix)

NORM         = 'l2'
SUBLINEAR_TF = True
LOWERCASE    = True
NGRAM_RANGE  = (1, 2)
SVM_C        = 1.0
MAX_ITER     = 2000
N_FOLDS      = 5
RESULTS_DIR  = 'results'
os.makedirs(RESULTS_DIR, exist_ok=True)
print('Setup complete ✅')

Setup complete ✅


---
## Dataset

In [2]:
spam_messages = [
    "Free entry in 2 a weekly competition to win FA Cup final tickets",
    "WINNER!! As a valued network customer you have been selected to receive a prize reward",
    "Had your mobile 11 months or more? U R entitled to Update to the latest colour mobiles",
    "SIX chances to win CASH! From 100 to 20000 pounds txt CSH11 and send to 87575",
    "URGENT! You have won a 1 week FREE membership in our prize draw",
    "Congratulations ur awarded 500 of bonus points. To claim call 09050000327",
    "You have been selected for a cash prize of 2000 pounds call now to claim",
    "FREE MESSAGE: Congratulations you have been awarded 1000 prize money",
    "Claim your free gift now! Text WIN to 87121 to collect",
    "You are a winner! Call 0800 to collect your prize today",
    "PRIVATE! Your 2003 Account Statement shows 800 prize GUARANTEED Call 09061743806",
    "Had your number for 11 months? You could be eligible to receive a new handset free",
    "Thanks for your subscription to Ringtone UK your mobile will be charged 5 per month",
    "This is the 2nd attempt to contact you! Re: claim 100 prize reward",
    "FREE ringtone waiting to be collected just text the word GET to 85023 now",
    "URGENT! Your mobile number has been awarded a 2000 Bonus Caller Prize to claim",
    "Save money on all your calls! Text SAVE to 83383 and get amazing deals",
    "You have won 1 million dollars! Call now to claim your free winnings",
    "Tone 2 go 4 wap. 16 tones to choose from at 1.50 each. Buy now",
    "Congratulations! You have been chosen to win a brand new iPhone. Click now",
    "FREE offer: Get 3 months of premium service absolutely free when you sign up",
    "ALERT: Your bank account has been compromised. Call us immediately to verify",
    "Win a PlayStation console! Text PLAY to 69911 and enter our prize competition",
    "Call now to claim your FREE voucher worth 500 pounds at any store nationwide",
    "Special offer exclusively for you! Get cash back on every purchase you make",
    "You have been randomly selected to win a luxury holiday for two people",
    "Urgent reply needed: You are eligible for a government tax rebate of 400",
    "Double your money instantly! Our investment scheme guarantees 200 percent returns",
    "Get a free loan today! No credit checks, no documents, apply now online",
    "Limited time: Buy now and get a second item absolutely FREE of charge",
    "Your exclusive reward is waiting! Log in now to claim your special bonus",
    "We have tried to contact you regarding your accident claim from 3 years ago",
    "Important notice: Your subscription is about to expire. Renew now for free",
    "Congratulations on being selected! Reply YES to claim your cash prize now",
    "You qualify for a free phone upgrade! Call 0800 to find out more today",
    "Mobile number awarded 1000 bonus cash prize. Text CLAIM to 89555 immediately",
    "Exclusive deal: Get 50 percent off your next purchase when you text DEAL now",
    "Last chance! Your unclaimed prize of 750 pounds expires in 24 hours only",
    "Ringtone service: You have been charged 3.00 per week unless you text STOP",
    "Free stock tip: Buy NOW before this company goes public and make big profits",
    "Your PPI claim may be worth thousands. Reply to find out what you are owed",
    "Hot babes waiting to chat! Call 09098272756 for a great time tonight",
    "Reply now to WIN a weeks holiday for 2 to Balearics. Over 18 only",
    "CLAIM YOUR FREE PRIZE NOW! Go to the link and collect your reward today",
    "We are trying to contact you. Our records show your number won a car",
    "Cash prize: Send your name and address to claim 500 by return text message",
    "You have a secret admirer! Find out who by texting CRUSH to 55555 now",
    "FREE entry for monthly draw: Text WIN to 80082 and win a cash prize",
    "Earn extra money from home! No experience needed. Text EARN to 78866 today",
    "Our new service lets you get fast cash loans in minutes with no questions",
    "Call me back when you get this free offer for new customers only",
    "Text back 4 a great time! Call me NOW on 09094646173 for amazing fun",
    "Hi babe I am in town this weekend fancy a drink text back to confirm",
    "You have 1 new voicemail! Call 121 to listen to your important message now",
    "SMS SERVICES: For latest offers on mobiles reply OFFER to this number today",
    "U have been selected 2 receiv a guaranteed 1000 cash or a 2000 prize",
    "Urgent! Your account needs verification. Text back your PIN to confirm identity",
    "Final reminder: Collect your winnings of 3175 or they will be forfeited forever",
    "Excellent news! You have been shortlisted for a business grant of 5000",
    "Your free gift from our company is waiting. Reply NOW to claim before midnight",
    "Get paid to take surveys! Earn up to 150 per day from the comfort of home",
    "NEW! Long distance UNLIMITED minutes plan. Text CALL to get your free upgrade",
    "You won our raffle! Reply with your bank account details to receive payment",
    "SALE: Up to 80 percent off designer goods. Call 0800 or visit us online now",
    "Act fast! Your pre-approved loan of 10000 pounds is waiting for your reply",
    "As a Vodafone customer you are entitled to a free upgrade. Call us today",
    "Your monthly mobile bill has been waived! Text CONFIRM to receive the credit",
    "Free phone insurance for 1 month. Text INSURE to 65555 to activate your policy",
    "Jackpot winner! Your number has been matched. Call 0906 to collect the prize",
    "ATTENTION! Your parcel could not be delivered. Click the link to reschedule",
    "You are selected to take part in our customer satisfaction survey for cash",
    "Hello! This is a final notice regarding the legal action against your account",
    "Investment opportunity of the year! Send 100 and get 1000 back guaranteed",
    "Your 2 free cinema tickets are waiting. To claim text FILM to 83383 today",
    "Win the ultimate luxury prize package worth 5000 pounds. Hurry, ends tonight",
]
ham_messages = [
    "I am going to try for 2 months ha ha only joking with you",
    "Do you know what Mallika Sherawat did for the Indian film industry",
    "Ok lar Joking wif u oni take it easy",
    "Fine if that is the way you feel that is the way it is",
    "England v Macedonia dont be late. Its 11 o clock in the morning silly",
    "Is that seriously how you spell his name? That looks wrong to me",
    "I HAVE A DATE ON SUNDAY WITH WILL finally things are looking up",
    "What do you want when I come back from the shops tonight?",
    "Please go ahead with the plan. I just wanted to be sure first",
    "I wont be going home until late tonight. Do not wait up for me",
    "Great! I hope you like your present. Let me know what you think",
    "Sorry my roommate had to see a doctor this morning so I was late",
    "Are you free now? Can we meet at the cafe for coffee today?",
    "I am on my way. Be there in about 10 minutes or so",
    "Hey just got your message. What exactly is going on with the plan",
    "Can you call me later? I am in a meeting right now and cannot talk",
    "Thanks for letting me know. See you tonight at the restaurant then",
    "I will be late for dinner. Start without me and I will be there soon",
    "Did you get my last message? Just checking you received it okay",
    "Happy birthday! Hope you have a really great day today celebrating",
    "Running late. Sorry, stuck in traffic on the motorway right now",
    "Just left the office. Should be home by about 7 or 8 tonight",
    "Can you pick up some milk on your way home from work please",
    "Good morning! How are you feeling today? Hope you are better",
    "Yes I am coming tonight. What time should I get there exactly?",
    "Have you eaten yet? I can make some food when I get home",
    "Thanks for the help yesterday. Really appreciated what you did",
    "No problem at all. Happy to help anytime you need it",
    "Where are you? We have been waiting for you for 30 minutes",
    "Call me when you get a chance. Need to talk to you about something",
    "I just got back from the gym. Pretty tired but feel good now",
    "What are you up to this weekend? Want to hang out and catch up?",
    "The meeting has been moved to 3pm. Please make sure you are there",
    "See you at the station at half past six. Do not be late this time",
    "I forgot to bring my lunch today. Going to have to buy something",
    "It was good to see you last night. We should do it again soon",
    "Let me know when you arrive and I will come down to meet you",
    "Can we rearrange for tomorrow? Something has come up today",
    "How is your mum doing? Hope she is feeling much better now",
    "Got your letter. Will reply properly when I get a chance later",
    "The kids are driving me crazy today. Cannot wait for bedtime",
    "Just checking you are okay. You seemed a bit quiet earlier today",
    "Love you too. Miss you loads. Cannot wait to see you next week",
    "Did you watch the match last night? What did you think of it?",
    "I am at the shops now. Do you need anything while I am here?",
    "Sorry I missed your call. Was on the other line. Will ring you back",
    "Heading to bed now. Speak tomorrow. Good night and sleep well",
    "Just to let you know I got here safely. All good at this end",
    "Still at work unfortunately. It has been a really long day today",
    "The traffic is terrible today. Going to be about 20 minutes late",
    "Good news! I got the job! Cannot believe it worked out so well",
    "I think I left my bag at your place. Can you check for me please",
    "On my way to the cinema. Starts at 8 so should be back by 10",
    "Just had the best meal ever. You have to try this restaurant",
    "Your parcel has arrived. It is at the front desk waiting for you",
    "We have run out of coffee. Can you grab some on the way home?",
    "I am not sure what to cook tonight. Any suggestions from you?",
    "Do not forget about mums birthday on Saturday. Have you got a gift",
    "Just woke up. Feel so much better than I did yesterday thankfully",
    "The meeting went really well. They seemed very impressed with us",
    "Can you send me the document before you leave the office today",
    "I will be in the area later. Want to grab a coffee and chat?",
    "Hope your interview goes well today. You will be amazing I know",
    "Thanks for dinner last night. It was absolutely delicious food",
    "My phone battery is dying. Will text you when I get to a charger",
    "So tired today. Did not sleep well at all last night, kept waking",
    "Just got to the hotel. It is really nice here. Room is lovely",
    "Are you watching the news? Something big seems to be happening",
    "I cannot find my keys anywhere. Have you seen them by any chance",
    "Just finished my exam. Think it went okay but not totally sure",
    "We should plan a holiday together this summer. What do you think",
    "The doctor said everything looks fine. Nothing to worry about",
    "I have to cancel tomorrow. Really sorry about the short notice",
    "Great to hear from you! It has been such a long time since we spoke",
]
texts_balanced  = spam_messages + ham_messages
labels_balanced = [1]*len(spam_messages) + [0]*len(ham_messages)
y_balanced      = np.array(labels_balanced)
print(f'Balanced dataset  : {len(texts_balanced)} samples — {labels_balanced.count(1)} spam, {labels_balanced.count(0)} ham')

Balanced dataset  : 149 samples — 75 spam, 74 ham


---
## Failure Scenario: Severe Class Imbalance

**Scenario:** The training data contains only **5.4% spam (8 messages) and 94.6% ham (141 messages)**, mimicking a real-world deployment setting where deceptive content is rare relative to genuine content.

**Why the method is expected to struggle:**  
Task 1.2 Assumption 1 states that the BIGRAMS+_SVM method assumes a balanced 50/50 class distribution. This assumption is baked into two specific design choices in Ott et al. (2011). First, the Naive Bayes formulation in Section 3.2 simplifies the posterior probability (Equation 1) into a likelihood ratio (Equation 2) by cancelling equal class priors — a step that is only valid when `P(deceptive) = P(truthful) = 0.5`. Second, the LinearSVC with default `C=1.0` does not apply any class-weight adjustment, so its hinge-loss objective treats each training example equally regardless of class frequency; when one class overwhelmingly outnumbers the other, the SVM finds it more cost-effective to classify everything as the majority class, since that already minimises total misclassification count. Under severe imbalance the accuracy metric becomes misleading — a degenerate classifier that predicts ham for every input achieves ~94.6% accuracy by default, making the class appear solved when in fact the minority class (spam/deceptive) is never detected at all. This scenario directly mirrors the real-world setting that Ott et al. acknowledge in Section 6: online review platforms contain far more genuine than fake reviews, meaning any deployed version of their classifier would face exactly this imbalance without corrective resampling or cost-sensitive training.

### Constructing the Imbalanced Dataset

In [3]:
# Construct severely imbalanced dataset
# 8 spam (5.4%) + 141 ham (94.6%) = 149 total
# Same total size as balanced dataset for a fair comparison
N_SPAM_IMB = 8    # ~5% of 149
N_HAM_IMB  = 141  # ~95% of 149

np.random.seed(RANDOM_SEED)
spam_idx = np.random.choice(len(spam_messages), N_SPAM_IMB, replace=False)
# ham_messages only has 74 unique entries; use replace=True to reach 141
ham_idx  = np.random.choice(len(ham_messages),  N_HAM_IMB,  replace=True)

texts_imb  = [spam_messages[i] for i in spam_idx] + [ham_messages[i] for i in ham_idx]
labels_imb = [1]*N_SPAM_IMB + [0]*N_HAM_IMB
y_imb      = np.array(labels_imb)

print(f'Imbalanced dataset: {len(texts_imb)} samples')
print(f'  Spam (deceptive): {N_SPAM_IMB}  ({N_SPAM_IMB/len(texts_imb)*100:.1f}%)')
print(f'  Ham  (truthful) : {N_HAM_IMB}  ({N_HAM_IMB/len(texts_imb)*100:.1f}%)')
print(f'Majority-class baseline accuracy: {N_HAM_IMB/len(texts_imb)*100:.1f}%')

Imbalanced dataset: 149 samples
  Spam (deceptive): 8  (5.4%)
  Ham  (truthful) : 141  (94.6%)
Majority-class baseline accuracy: 94.6%


**What the code does:**  
The imbalanced dataset is constructed by sampling 8 spam messages (without replacement) and 141 ham messages (with replacement, since there are only 74 unique ham messages) from the original balanced pool. The total sample size is held constant at 149 to ensure differences between balanced and imbalanced results are driven by class distribution rather than dataset size. The majority-class baseline — accuracy achievable by always predicting ham — is 94.6%, which will serve as an important reference point when interpreting results.

---
### Experiment: Full Method on Balanced vs Imbalanced Data

In [4]:
scoring = {
    'accuracy' : 'accuracy',
    'f1'       : make_scorer(f1_score,        zero_division=0),
    'recall'   : make_scorer(recall_score,    zero_division=0),
    'precision': make_scorer(precision_score, zero_division=0),
}
cv5 = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_SEED)

def run_full(texts_in, y_in):
    vec = TfidfVectorizer(ngram_range=NGRAM_RANGE, norm=NORM,
                          sublinear_tf=SUBLINEAR_TF, lowercase=LOWERCASE)
    X   = vec.fit_transform(texts_in)
    clf = LinearSVC(C=SVM_C, max_iter=MAX_ITER, random_state=RANDOM_SEED)
    return cross_validate(clf, X, y_in, cv=cv5, scoring=scoring)

res_bal = run_full(texts_balanced, y_balanced)
res_imb = run_full(texts_imb,      y_imb)

print('=== Balanced (50% spam / 50% ham) ===')
print(f'  Accuracy : {res_bal["test_accuracy"].mean()*100:.1f}%')
print(f'  Precision: {res_bal["test_precision"].mean()*100:.1f}%')
print(f'  Recall   : {res_bal["test_recall"].mean()*100:.1f}%')
print(f'  F1 Score : {res_bal["test_f1"].mean()*100:.1f}%')
print()
print('=== Imbalanced (5.4% spam / 94.6% ham) ===')
print(f'  Accuracy : {res_imb["test_accuracy"].mean()*100:.1f}%  ← misleadingly high')
print(f'  Precision: {res_imb["test_precision"].mean()*100:.1f}%')
print(f'  Recall   : {res_imb["test_recall"].mean()*100:.1f}%  ← ZERO: no spam detected')
print(f'  F1 Score : {res_imb["test_f1"].mean()*100:.1f}%  ← ZERO: complete failure')
print(f'  Per-fold recall: {[round(x*100,1) for x in res_imb["test_recall"]]}')

=== Balanced (50% spam / 50% ham) ===
  Accuracy : 98.0%
  Precision: 98.7%
  Recall   : 97.3%
  F1 Score : 98.0%

=== Imbalanced (5.4% spam / 94.6% ham) ===
  Accuracy : 94.6%  ← misleadingly high
  Precision: 0.0%
  Recall   : 0.0%  ← ZERO: no spam detected
  F1 Score : 0.0%  ← ZERO: complete failure
  Per-fold recall: [np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0)]


**What the code does:**  
The same full BIGRAMS+_SVM pipeline (TF-IDF with `ngram_range=(1,2)`, LinearSVC with `C=1.0`) is evaluated via `cross_validate` with four metrics on both datasets. `StratifiedKFold` is used to ensure each fold preserves the imbalanced ratio, meaning each test fold contains approximately 1 spam and 28 ham messages. Four metrics are collected rather than just accuracy: the contrast between near-100% accuracy and 0% recall/F1 on the imbalanced data is the central empirical finding.

---
### Visualisation

In [5]:
from sklearn.model_selection import train_test_split

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor('#FAFAFA')

# Left panel: all-metrics comparison bar chart
ax = axes[0]
metric_names  = ['Accuracy', 'Precision', 'Recall', 'F1 Score']
bal_vals  = [res_bal['test_accuracy'].mean()*100,  res_bal['test_precision'].mean()*100,
             res_bal['test_recall'].mean()*100,    res_bal['test_f1'].mean()*100]
imb_vals  = [res_imb['test_accuracy'].mean()*100,  res_imb['test_precision'].mean()*100,
             res_imb['test_recall'].mean()*100,    res_imb['test_f1'].mean()*100]
x = np.arange(len(metric_names)); w = 0.35
b1 = ax.bar(x-w/2, bal_vals, w, color='#2196F3',
            label='Balanced (50% spam, 50% ham)', edgecolor='white', zorder=3)
b2 = ax.bar(x+w/2, imb_vals, w, color='#E53935',
            label='Imbalanced (5.4% spam, 94.6% ham)', edgecolor='white', zorder=3)
for bar in list(b1)+list(b2):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5,
            f'{bar.get_height():.1f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
# Annotate catastrophic F1 drop
ax.annotate('', xy=(x[3]+w/2, imb_vals[3]+2),
            xytext=(x[3]+w/2, bal_vals[3]-2),
            arrowprops=dict(arrowstyle='->', color='black', lw=2))
ax.text(x[3]+w/2+0.1, (imb_vals[3]+bal_vals[3])/2,
        '\u221298pp\ncatastrophic\nfailure', fontsize=8, color='#B71C1C', fontweight='bold')
ax.set_xticks(x); ax.set_xticklabels(metric_names, fontsize=10)
ax.set_ylim(0, 115); ax.set_ylabel('Score (%)', fontsize=11)
ax.set_title('Balanced vs Imbalanced Dataset\nAll Metrics Comparison', fontsize=11, fontweight='bold')
ax.legend(fontsize=9); ax.set_facecolor('#F5F5F5')
ax.grid(axis='y', alpha=0.4, zorder=0)
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

# Right panel: confusion matrix on held-out imbalanced test set
vec_imb = TfidfVectorizer(ngram_range=NGRAM_RANGE, norm=NORM,
                           sublinear_tf=SUBLINEAR_TF, lowercase=LOWERCASE)
X_imb_full = vec_imb.fit_transform(texts_imb)
X_tr, X_te, y_tr, y_te = train_test_split(
    X_imb_full, y_imb, test_size=0.25, stratify=y_imb, random_state=RANDOM_SEED)
clf_imb = LinearSVC(C=SVM_C, max_iter=MAX_ITER, random_state=RANDOM_SEED)
clf_imb.fit(X_tr, y_tr)
y_te_pred = clf_imb.predict(X_te)
cm = confusion_matrix(y_te, y_te_pred)

ax2 = axes[1]
sns.heatmap(cm, annot=True, fmt='d', cmap='Reds', ax=ax2,
            xticklabels=['Ham\n(predicted)', 'Spam\n(predicted)'],
            yticklabels=['Ham\n(true)', 'Spam\n(true)'],
            annot_kws={'size':14, 'weight':'bold'}, linewidths=0.5, cbar_kws={'shrink':0.8})
ax2.set_title('Confusion Matrix — Imbalanced Test Set\n(classifier predicts ALL as ham)',
              fontsize=11, fontweight='bold')
ax2.set_xlabel('Predicted Label', fontsize=11); ax2.set_ylabel('True Label', fontsize=11)

plt.suptitle('Task 3.2 — Failure Mode: Severe Class Imbalance\n'
             'Assumption 1 Violation (Section 3.2, Ott et al. 2011)',
             fontsize=12, fontweight='bold', y=1.02)
plt.tight_layout()
VIZ = os.path.join(RESULTS_DIR, 'task_3_2_failure_mode.png')
plt.savefig(VIZ, dpi=150, bbox_inches='tight', facecolor='#FAFAFA')
plt.show()
print(f'Saved to: {VIZ} ✅')
print(f'\nConfusion matrix (imbalanced held-out test):\n{cm}')

Saved to: results/task_3_2_failure_mode.png ✅

Confusion matrix (imbalanced held-out test):
[[36  0]
 [ 2  0]]


**What this figure shows:**  
The left panel makes the deceptive gap between accuracy and F1 visually immediate: both bars look similar for accuracy (98.0% vs 94.6%), but the imbalanced dataset collapses to 0% for precision, recall, and F1. The right panel provides the mechanistic explanation — the confusion matrix shows the classifier classifying every single test example as ham, including the 2 true spam messages in the test set. This is the hallmark of the majority-class collapse failure: the SVM finds that predicting ham universally is globally optimal under the imbalanced hinge loss, and so it never activates the spam decision region at all.

---
## Failure Explanation

The failure observed here — 0% recall and 0% F1 despite 94.6% accuracy — is a direct consequence of violating **Assumption 1** from Task 1.2: that the class distribution is balanced at 50/50. Ott et al. (2011) explicitly constructed their dataset to be balanced (400 deceptive + 400 truthful, Section 3.2), and this design choice is not a mere convenience — it is a requirement for their evaluation to be meaningful. With only 8 spam messages out of 149 (5.4%), the LinearSVC's hinge loss finds it globally cost-minimal to classify everything as the majority class (ham), since this already achieves ~94.6% accuracy and incurs almost no misclassification penalty. The SVM's weight vector `w` converges to a direction that is dominated by the ham class's TF-IDF distribution, and the decision boundary `w·x + b = 0` is positioned such that all test points fall on the ham side — exactly the pattern visible in the confusion matrix (all predictions are ham, zero spam correctly identified). Accuracy, the metric used in the paper's Table 3, is entirely uninformative under this imbalance: it rewards the degenerate always-predict-majority strategy, making the classifier appear to work when it has completely failed to learn the minority class. This reveals a fundamental limitation in the paper's evaluation methodology — reporting only accuracy on a balanced dataset gives a misleadingly optimistic picture of real-world deployability, where platforms like TripAdvisor would have far fewer fake reviews than genuine ones.

**Suggested fix:** Replace the standard LinearSVC with a **class-weight-adjusted SVC** using `class_weight='balanced'`, which scales the hinge loss penalty for each class inversely proportional to its frequency, forcing the model to treat minority-class errors as more costly and preventing the majority-class collapse.

**Paper reference:** Section 3.2 (balanced dataset construction), Section 6 (limitations and future work acknowledging deployment challenges on real platforms with unknown class priors).